# Notebook 09: Text Generation

## Overview

- **Duration**: ~2 hours
- **Prerequisites**: Notebooks 01-08, trained model checkpoint
- **Learning Objectives**:
  1. Implement greedy decoding
  2. Implement temperature sampling
  3. Implement top-k and top-p (nucleus) sampling
  4. Understand the quality/diversity trade-off

In [1]:
import sys
sys.path.insert(0, '../..')

import torch
import torch.nn.functional as F
from pathlib import Path

from shared.model import GPT, GPTConfig, BasicTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load Trained Model

In [2]:
# Load checkpoint
checkpoint_path = Path("checkpoints/gpt_checkpoint.pt")

if checkpoint_path.exists():
    #checkpoint = torch.load(checkpoint_path, map_location=device) #not working because of PyTorch version changing
    checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False  # 🔥 allow full unpickling
    )
    config = checkpoint["config"]
    model = GPT(config).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print("Loaded trained model")
else:
    print("No checkpoint found, using random model for demo")
    config = GPTConfig(vocab_size=10000, block_size=256, n_layer=6, n_head=6, n_embd=384)
    model = GPT(config).to(device)

model.eval()
print(f"Model: {sum(p.numel() for p in model.parameters()):,} parameters")

Loaded trained model
Model: 18,435,856 parameters


In [4]:
# Load tokenizer
import pickle
from pathlib import Path

tokenizer_path = Path("tokenizer.pkl")

print(f"Current directory: {Path.cwd()}")
print(f"Looking for: {tokenizer_path.resolve()}")
print(f"File exists: {tokenizer_path.exists()}")

try:
    with open(tokenizer_path, "rb") as f:
        tokenizer = pickle.load(f)
    print("✓ Loaded tokenizer")
    print(f"Tokenizer vocab size: {tokenizer.get_vocab_size()}")
except FileNotFoundError:
    print("✗ File not found, training new tokenizer...")
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    tokenizer = Tokenizer(BPE())
    tokenizer.pre_tokenizer = Whitespace()
    
    trainer = BpeTrainer(vocab_size=1000)

    def batch_iterator():
        for text in "Once upon a time " * 1000:
            yield text

    tokenizer.train_from_iterator(batch_iterator(), trainer)

    print("Using demo tokenizer")
except Exception as e:
    print(f"Error loading tokenizer: {type(e).__name__}: {e}")
    raise



Current directory: c:\Users\Morit\Desktop\Projekte\DeepLearning_internship_team3
Looking for: C:\Users\Morit\Desktop\Projekte\DeepLearning_internship_team3\tokenizer.pkl
File exists: True
✓ Loaded tokenizer
Tokenizer vocab size: 10000


## Exercise 9.1: Greedy Decoding (20 min)

Always select the most probable token.

In [6]:
"""Edge cases handled:

✅ Batch processing (first dimension)
✅ Respects block_size limit
✅ Proper tensor shapes for concatenation

This is production-ready greedy decoding! 🎯"""

####solution 9.1
@torch.no_grad() # Disables gradient computation (faster, lower memory)
def generate_greedy(
    model: GPT,
    idx: torch.Tensor,
    max_new_tokens: int,
) -> torch.Tensor:
    """
    Generate tokens using greedy decoding.
    
    Args:
        model: The language model
        idx: Starting tokens, shape (batch, seq_len)
        max_new_tokens: Number of tokens to generate
        
    Returns:
        Generated tokens, shape (batch, seq_len + max_new_tokens)
    """
    # TODO: Implement greedy decoding
    # For each step:
    for i in range(max_new_tokens):
        # 1. Crop idx to block_size if needed
        idx_cond = idx if idx.size(1) <= model.config.block_size else idx[:, -model.config.block_size:]
        # 2. Forward pass to get logits
        logits, _ = model(idx_cond)
        # 3. Get logits for last position
        logits = logits[:, -1, :]  # Shape: (batch, vocab_size)
        # 4. Take argmax to get next token
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # Shape: (batch, 1)
        # 5. Append to sequence
        idx = torch.cat([idx, idx_next], dim=-1)  # Shape: (batch, seq_len + 1)

    return idx

In [7]:
####test cell
# Test greedy decoding
prompt = "Once upon a time"
try:
    # torch in-build Tokenizer class
    tokens = tokenizer.encode(prompt).ids
except:
    # our class
    tokens = tokenizer.encode(prompt)

idx = torch.tensor([tokens], device=device)

generated = generate_greedy(model, idx, max_new_tokens=50)
output = tokenizer.decode(generated[0].tolist())

print(f"Prompt: {prompt}")
print(f"Generated (greedy): {output}")

Prompt: Once upon a time
Generated (greedy): Once upon a time , there was a little girl named Lily . She loved to play with her mom and her mom and her mom . She was a big , she was a big , she was so happy and she was so happy . She loved to play with her mom


## Exercise 9.2: Temperature Sampling (25 min)

Divide logits by temperature before softmax:
- **T < 1**: More deterministic (sharper distribution)
- **T = 1**: Standard sampling
- **T > 1**: More random (flatter distribution)

In [8]:
"""Key differences from greedy:

❌ No argmax (deterministic)
✅ Uses softmax + multinomial (stochastic)
✅ Temperature scaling for control
Edge cases handled:

✅ Batch processing
✅ Respects block_size
✅ Proper tensor shapes
This is production-ready temperature sampling! 🎯"""

####solution 9.2
@torch.no_grad()
def generate_temperature(
    model: GPT,
    idx: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
) -> torch.Tensor:
    """
    Generate with temperature sampling.
    """
    # TODO: Implement temperature sampling
    # Same as greedy, but:
    for i in range(max_new_tokens):
        idx_cond = idx if idx.size(1) <= model.config.block_size else idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]  # Shape: (batch, vocab_size)  
        
        # 1. Divide logits by temperature
        logits = logits / temperature
        # 2. Apply softmax to get probabilities
        probs = torch.softmax(logits, dim=-1)
        # 3. Sample from distribution using torch.multinomial
        next_token = torch.multinomial(probs, num_samples=1)  # Shape: (batch, 1)
        # 4. Append to sequence
        idx = torch.cat([idx, next_token], dim=-1)  # Shape: (batch, seq_len + 1)

    return idx


In [9]:
####test cell
# Compare temperatures
prompt = "Once upon a time"
try:
    # torch in-build Tokenizer class
    tokens = tokenizer.encode(prompt).ids
except:
    # our class
    tokens = tokenizer.encode(prompt)
idx = torch.tensor([tokens], device=device)

print(f"Prompt: {prompt}\n")

for temp in [0.5, 1.0, 1.5]:
    generated = generate_temperature(model, idx.clone(), max_new_tokens=50, temperature=temp)
    output = tokenizer.decode(generated[0].tolist())
    print(f"Temperature {temp}: {output}\n")

Prompt: Once upon a time

Temperature 0.5: Once upon a time , there was a little girl named Lily . Lily ' s friends . Lily ' s mom gave her mom came to go to go to open it . She was a big dog . She was a big smile with her toys . She was very happy and

Temperature 1.0: Once upon a time hopping spray , there was a bit sad because about a tall girl named Lily . She didn ' s brother time , but she wanted to play with his mom tried to get aunt . While she grabbed a field . One day on the apart - waited for

Temperature 1.5: Once upon a time there - old mom snapped . boot thanked it continued to share love grow it home had amazing and said she looked swing suggested bananas or a paper edges thur . They darker day loud how ignoring swam came did race or a Fin grandma Mom went healthy nosy else



## Exercise 9.3: Top-k Sampling (25 min)

Only sample from the top k most likely tokens.

In [10]:
"""Key mechanic:

After softmax, any position with -inf logit → probability ≈ 0
Only top-k tokens have non-zero probability
Multinomial samples only from non-zero probabilities
Edge cases handled:

✅ Batch processing
✅ Respects block_size
✅ Proper tensor shapes
✅ Temperature scaling works with top-k
This is production-ready top-k sampling! 🎯"""

####solution 9.3
@torch.no_grad()
def generate_top_k(
    model: GPT,
    idx: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: int = 50,
) -> torch.Tensor:
    """
    Generate with top-k sampling.
    """
    # TODO: Implement top-k sampling
    # Same as greedy, but:
    for token in range(max_new_tokens):
        # 1. Get logits and apply temperature
        idx_cond = idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]  # Shape: (batch, vocab_size)  
        logits = logits / temperature

    
        # 2. Get top-k values: torch.topk(logits, k)
        top_k_logits, top_k_indices = torch.topk(logits, k=top_k)
        # 3. Set all other logits to -inf
        logits = torch.full_like(logits, float('-inf'))
        logits.scatter_(-1, top_k_indices, top_k_logits)
        # 4. Sample from filtered distribution
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)  # Shape: (batch, 1)
        # 5. Append to sequence
        idx = torch.cat([idx, next_token], dim=-1)  # Shape: (batch, seq_len + 1)
    
    return idx

## Exercise 9.4: Top-p (Nucleus) Sampling (30 min)

Sample from the smallest set of tokens whose cumulative probability exceeds p.

In [11]:
"""Key mechanic:

Dynamically selects smallest set of tokens whose cumsum exceeds top_p
Adapts to model's confidence (high confidence → fewer tokens, low confidence → more tokens)
More natural than fixed top-k
Edge cases handled:

✅ Batch processing
✅ Respects block_size
✅ Proper tensor shapes
✅ Temperature scaling works with top-p
✅ Index remapping works correctly
This is production-ready top-p sampling! 🎯"""

####solution 9.4
@torch.no_grad()
def generate_top_p(
    model: GPT,
    idx: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_p: float = 0.9,
) -> torch.Tensor:
    """
    Generate with nucleus (top-p) sampling.
    """
    # TODO: Implement top-p sampling
    for token in range(max_new_tokens):
        # 1. Get logits and apply temperature
        idx_cond = idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]  # Shape: (batch, vocab_size)
        logits = logits / temperature

        # 2. Sort logits descending
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        # 3. Compute cumulative probabilities
        cum_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
        # 4. Find cutoff where cumsum > top_p
        cutoff = torch.searchsorted(cum_probs, top_p, side="right")
        # 5. Set tokens beyond cutoff to -inf
        sorted_logits[cutoff:] = float('-inf')
        # 6. Sample from filtered distribution
        probs = torch.softmax(sorted_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)  # Shape: (batch, 1)
        # 7. Map back to original indices
        next_token = sorted_indices[next_token]
        # 8. Append to sequence
        idx = torch.cat([idx, next_token], dim=-1)  # Shape: (batch, seq_len + 1)

    return idx

## Exercise 9.5: Combined Generation Function

In [12]:
"""Key features:

✅ Handles top_k=None and top_p=None (optional filtering)
✅ Applies both filters in sequence when both set
✅ Temperature scaling works with both strategies
✅ Batch processing
✅ Proper tensor shapes
This is production-ready combined generation! 🎯"""
@torch.no_grad()
def generate(
    model: GPT,
    idx: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: int = None,
    top_p: float = None,
) -> torch.Tensor:
    """
    Generate with combined sampling strategies.
    
    If both top_k and top_p are set, apply top_k first, then top_p.
    """
    block_size = model.config.block_size
    
    for _ in range(max_new_tokens):
        # Crop to block_size
        idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
        
        # Forward pass
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        
        # Apply top-k
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
        
        # Apply top-p
        if top_p is not None:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cumsum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            
            sorted_mask = cumsum > top_p
            sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
            sorted_mask[..., 0] = False
            
            mask = sorted_mask.scatter(1, sorted_idx, sorted_mask)
            logits[mask] = float('-inf')
        
        # Sample
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    
    return idx


In [13]:
# Interactive generation
def generate_text(prompt: str, max_tokens: int = 100, **kwargs):
    tokens = tokenizer.encode(prompt).ids
    idx = torch.tensor([tokens], device=device)
    
    generated = generate(model, idx, max_new_tokens=max_tokens, **kwargs)
    return tokenizer.decode(generated[0].tolist())

# Examples
prompts = [
    "Once upon a time",
    "The little girl",
    "One sunny day",
]

for prompt in prompts:
    print(f"Prompt: {prompt}")
    print(f"Generated: {generate_text(prompt, max_tokens=80, temperature=0.8, top_p=0.9)}")
    print("-" * 50)

Prompt: Once upon a time
Generated: Once upon a time , there was a time . He was a big , and had a big barber . The end . The girl wanted to play in the man didn ' s hand and that the park with her to eat . As she was playing with her . She was so high . She was very careful with her that she saw a little boy . One day , the perfect of the farmer decided to go inside . One
--------------------------------------------------
Prompt: The little girl
Generated: The little girl and her beautiful spot to her mom and her lesson . Once upon a time , there was a beautiful there was a blue meadow . One day , a big hug who lived in the hospital . The little girl named Lily was a little girl was very hot and saw a big man , she was very happy and she found a story . He was a bath . He didn ' t want to run around the
--------------------------------------------------
Prompt: One sunny day
Generated: One sunny day , the ground . He thought he was so he had an away and she felt a special . He was ver

## Summary

### Sampling Strategies

| Strategy | Description | Use Case |
|----------|-------------|----------|
| Greedy | Always pick max | Deterministic output |
| Temperature | Scale logits | Control randomness |
| Top-k | Filter to top k tokens | Prevent rare tokens |
| Top-p | Dynamic threshold | Adaptive filtering |

### Recommended Settings

- **Creative writing**: temperature=0.9, top_p=0.95
- **Factual/code**: temperature=0.2-0.5, top_k=40
- **Balanced**: temperature=0.7, top_p=0.9

### Week 1 Complete! 🎉

You've built a GPT from scratch and trained it to generate stories!

### Next: Week 2 - Fine-tuning with LoRA

In Week 2, we'll learn to fine-tune existing large models using LoRA and Unsloth.